In [ ]:
"""
Script : Redistribution de fact_sales sur 2018-2022
- Saisonnalité mensuelle
- Tendance de croissance annuelle (+8%/an)
- Creux COVID Q2 2020
- UPDATE direct PostgreSQL par batch
"""

import psycopg2
import psycopg2.extras
import random
import math
from datetime import date, timedelta
from collections import defaultdict

# ─── CONFIG ──────────────────────────────────────────────────────────────────
DB_CONFIG = {
    "dbname":   "grocery_db",
    "user":     "postgres",
    "password": "2002",   # ← adapter si nécessaire
    "host":     "localhost",
    "port":     5432,
}
BATCH_SIZE = 50_000
RANDOM_SEED = 42

# ─── POIDS ANNUELS (somme = 1.0) ─────────────────────────────────────────────
YEAR_WEIGHTS = {
    2018: 0.15,
    2019: 0.22,
    2020: 0.12,   # creux COVID
    2021: 0.25,
    2022: 0.26,
}

# ─── SAISONNALITÉ MENSUELLE (index relatif, sera normalisé) ──────────────────
MONTH_SEASON = {
     1: 0.80,   # Janvier  : creux post-fêtes
     2: 0.85,
     3: 0.90,
     4: 0.95,
     5: 1.00,
     6: 1.10,   # Été
     7: 1.15,
     8: 1.15,
     9: 1.05,
    10: 1.00,
    11: 1.10,   # Black Friday
    12: 1.30,   # Fêtes
}

# ─── COEFFICIENT PRIX (croissance annuelle +8%/an base 2018) ─────────────────
def price_coeff(year: int, month: int) -> float:
    growth = 1.08 ** (year - 2018)
    # COVID Q2 2020 : légère réduction de prix (-10%)
    if year == 2020 and month in (3, 4, 5, 6):
        growth *= 0.90
    return growth

# ─── COEFFICIENT VOLUME (quantity) ───────────────────────────────────────────
def volume_coeff(year: int, month: int) -> float:
    """COVID Q2 2020 : -40% volume"""
    if year == 2020 and month in (3, 4, 5, 6):
        return 0.60
    return 1.0

# ─── GÉNÉRATION DU CALENDRIER PONDÉRÉ ────────────────────────────────────────
def build_date_pool() -> list[date]:
    """Retourne une liste de toutes les dates 2018-2022 pondérées."""
    pool = []
    for year in range(2018, 2023):
        for month in range(1, 13):
            # Nombre de jours dans le mois
            if month == 12:
                days = (date(year + 1, 1, 1) - date(year, 12, 1)).days
            else:
                days = (date(year, month + 1, 1) - date(year, month, 1)).days

            # Poids = poids annuel × saisonnalité mensuelle × jours du mois
            w = YEAR_WEIGHTS[year] * MONTH_SEASON[month] * days

            for d in range(1, days + 1):
                pool.append((date(year, month, d), w))
    return pool

def sample_dates(pool: list, n: int, rng: random.Random) -> list[date]:
    """Tire n dates depuis le pool pondéré."""
    dates_list  = [x[0] for x in pool]
    weights_list = [x[1] for x in pool]
    return rng.choices(dates_list, weights=weights_list, k=n)

# ─── RÉCUPÉRATION DES IDs ─────────────────────────────────────────────────────
def fetch_all_rows(conn):
    """Retourne [(salesid, quantity, totalprice)] pour toutes les lignes."""
    print("📥 Chargement des salesid, quantity, totalprice…")
    rows = []
    with conn.cursor(name="big_cursor") as cur:
        cur.itersize = 100_000
        cur.execute("SELECT salesid, quantity, totalprice FROM fact_sales ORDER BY salesid")
        while True:
            chunk = cur.fetchmany(100_000)
            if not chunk:
                break
            rows.extend(chunk)
            print(f"   … {len(rows):,} lignes lues", end="\r")
    print(f"\n   → {len(rows):,} lignes chargées")
    return rows

# ─── UPDATE PAR BATCH ─────────────────────────────────────────────────────────
def update_batch(conn, batch: list):
    """
    batch : list de (new_date, new_quantity, new_totalprice, salesid)
    """
    sql = """
        UPDATE fact_sales AS f
        SET
            date         = v.new_date,
            quantity     = v.new_qty,
            totalprice   = v.new_price
        FROM (VALUES %s) AS v(new_date, new_qty, new_price, sid)
        WHERE f.salesid = v.sid
    """
    with conn.cursor() as cur:
        psycopg2.extras.execute_values(
            cur, sql, batch,
            template="(%s::date, %s::integer, %s::numeric, %s::bigint)",
            page_size=BATCH_SIZE,
        )
    conn.commit()

# ─── MAIN ─────────────────────────────────────────────────────────────────────
def main():
    rng = random.Random(RANDOM_SEED)

    print("🔌 Connexion PostgreSQL…")
    conn = psycopg2.connect(**DB_CONFIG)

    # 1. Récupérer toutes les lignes
    rows = fetch_all_rows(conn)
    n = len(rows)

    # 2. Générer le pool de dates pondéré
    print("📅 Construction du pool de dates…")
    pool = build_date_pool()

    # 3. Tirer n dates aléatoires pondérées
    print(f"🎲 Tirage de {n:,} dates…")
    new_dates = sample_dates(pool, n, rng)

    # 4. Construire les tuples de mise à jour
    print("⚙️  Calcul des nouveaux prix et quantités…")
    batch = []
    total_batches = math.ceil(n / BATCH_SIZE)
    batch_num = 0

    for i, (salesid, qty, price, new_date) in enumerate(
        zip([r[0] for r in rows],
            [r[1] for r in rows],
            [r[2] for r in rows],
            new_dates)
    ):
        y, m = new_date.year, new_date.month
        pc = price_coeff(y, m)
        vc = volume_coeff(y, m)

        # Appliquer un bruit léger ±3% pour éviter les valeurs trop régulières
        noise_p = rng.uniform(0.97, 1.03)
        noise_q = rng.uniform(0.97, 1.03)

        new_qty   = max(1, round(qty   * vc * noise_q))
        new_price = round(float(price) * pc * noise_p, 2)

        batch.append((new_date, new_qty, new_price, salesid))

        if len(batch) >= BATCH_SIZE:
            batch_num += 1
            print(f"   💾 Batch {batch_num}/{total_batches} — UPDATE {len(batch):,} lignes…")
            update_batch(conn, batch)
            batch.clear()

    # Dernier batch
    if batch:
        batch_num += 1
        print(f"   💾 Batch {batch_num}/{total_batches} — UPDATE {len(batch):,} lignes…")
        update_batch(conn, batch)

    conn.close()
    print("\n✅ Terminé ! Vérification rapide suggérée :")
    print("   SELECT date_part('year', date) AS year,")
    print("          COUNT(*) AS nb_ventes,")
    print("          ROUND(AVG(totalprice), 2) AS prix_moyen")
    print("   FROM fact_sales")
    print("   GROUP BY 1 ORDER BY 1;")

if __name__ == "__main__":
    main()

🔌 Connexion PostgreSQL…
📥 Chargement des salesid, quantity, totalprice…
   … 6,690,599 lignes lues
   → 6,690,599 lignes chargées
📅 Construction du pool de dates…
🎲 Tirage de 6,690,599 dates…
⚙️  Calcul des nouveaux prix et quantités…
   💾 Batch 1/134 — UPDATE 50,000 lignes…
   💾 Batch 2/134 — UPDATE 50,000 lignes…
   💾 Batch 3/134 — UPDATE 50,000 lignes…
   💾 Batch 4/134 — UPDATE 50,000 lignes…
   💾 Batch 5/134 — UPDATE 50,000 lignes…
   💾 Batch 6/134 — UPDATE 50,000 lignes…
   💾 Batch 7/134 — UPDATE 50,000 lignes…
   💾 Batch 8/134 — UPDATE 50,000 lignes…
   💾 Batch 9/134 — UPDATE 50,000 lignes…
   💾 Batch 10/134 — UPDATE 50,000 lignes…
   💾 Batch 11/134 — UPDATE 50,000 lignes…
   💾 Batch 12/134 — UPDATE 50,000 lignes…
   💾 Batch 13/134 — UPDATE 50,000 lignes…
   💾 Batch 14/134 — UPDATE 50,000 lignes…
   💾 Batch 15/134 — UPDATE 50,000 lignes…
   💾 Batch 16/134 — UPDATE 50,000 lignes…
   💾 Batch 17/134 — UPDATE 50,000 lignes…
   💾 Batch 18/134 — UPDATE 50,000 lignes…
   💾 Batch 19/134 